# 📈 Modelagem: Support Vector Regressor (SVR)

Este notebook foca na implementação do SVR. Como o SVM possui complexidade computacional elevada, **realizamos uma amostragem** do conjunto de desenvolvimento, mantendo o processo de validação oficial sobre a amostra.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

arquivo = Path("dados/processed/enem_2023_lc_c07.parquet")
df = pd.read_parquet(arquivo)

### Preparação e Amostragem Controlada
Para evitar que o kernel do Colab falhe, amostraremos aleatoriamente parte do conjunto de desenvolvimento e de teste.

In [ ]:
# Limitando tamanho devido à complexidade do SVR
TAMANHO_MAX = 20000 

dev_total = df.loc[df["CONJUNTO"] == "desenvolvimento"].reset_index(drop=True)
teste_total = df.loc[df["CONJUNTO"] == "teste"].reset_index(drop=True)

# Amostragem fixando a seed
dev = dev_total.sample(n=min(TAMANHO_MAX, len(dev_total)), random_state=42).reset_index(drop=True)
teste = teste_total.sample(n=min(TAMANHO_MAX, len(teste_total)), random_state=42).reset_index(drop=True)

X = dev.drop(columns=["NU_INSCRICAO", "CONJUNTO", "DOBRA", "NU_NOTA_LC"])
y = dev["NU_NOTA_LC"]
X_teste = teste.drop(columns=["NU_INSCRICAO", "CONJUNTO", "DOBRA", "NU_NOTA_LC"])
y_teste = teste["NU_NOTA_LC"]

# Recalculando dobras em cima da amostra atual
dobras = [
    (np.flatnonzero(dev["DOBRA"].to_numpy() != k), np.flatnonzero(dev["DOBRA"].to_numpy() == k)) 
    for k in (1, 2, 3)
]

### Pipeline com Necessidade de Escalabilidade (StandardScaler)
Diferente da árvore, o SVM não consegue calcular distâncias hiperplanas se as variáveis não estiverem padronizadas.

In [ ]:
ordens = {
    "Q006": list("ABCDEFGHIJKLMNOPQ"), "Q001": list("ABCDEFGH"),
    "Q002": list("ABCDEFGH"), "Q024": list("ABCDE"), "Q025": list("AB"),
    "ESCOLARIDADE_MAX_RESP": list("ABCDEFG") + ["INDETERMINADA"],
    "TP_FAIXA_ETARIA": [str(i) for i in range(1, 21)]
}

ord = [c for c in ordens if c in X.columns]
cat = [c for c in X.select_dtypes(include=["object", "category"]).columns if c not in ord]
num = [c for c in X.columns if c not in ord + cat]

# 1. Transformações Categóricas
steps = []
if ord: steps.append(("ord", OrdinalEncoder(categories=[ordens[c] for c in ord], handle_unknown="use_encoded_value", unknown_value=-1), ord))
if cat: steps.append(("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat))
if num: steps.append(("num", SimpleImputer(strategy="median", add_indicator=True), num))

# 2. Pipeline com o StandardScaler incluso
modelo_svm = Pipeline([
    ("preprocessamento", ColumnTransformer(steps)),
    ("scaler", StandardScaler(with_mean=False)), # with_mean=False ajuda se gerar matrizes esparsas
    ("modelo", SVR(kernel='rbf', C=1.0, epsilon=0.1))
])

### Validação e Resultados Finais
Note que a validação cruzada pode levar consideravelmente mais tempo neste modelo.

In [ ]:
# 1. Validação Cruzada
print("Iniciando Validação Cruzada do SVM...")
inicio_cv = time.time()
resultados_cv = cross_validate(
    modelo_svm, X, y, cv=dobras,
    scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"},
    n_jobs=1
)
tempo_cv = time.time() - inicio_cv
print("RESULTADOS — VALIDAÇÃO CRUZADA OFICIAL")
print(f"MAE médio:  {-resultados_cv['test_MAE'].mean():.2f}")
print(f"RMSE médio: {-resultados_cv['test_RMSE'].mean():.2f}")
print(f"R² médio:    {resultados_cv['test_R2'].mean():.4f}")
print(f"Tempo:       {tempo_cv / 60:.2f} minutos\n")

# 2. Treinamento Final no Teste
inicio_final = time.time()
modelo_svm.fit(X, y) 
tempo_final = time.time() - inicio_final

previsoes_teste = modelo_svm.predict(X_teste)
mae_teste = mean_absolute_error(y_teste, previsoes_teste)
rmse_teste = np.sqrt(mean_squared_error(y_teste, previsoes_teste))
r2_teste = r2_score(y_teste, previsoes_teste)

print("RESULTADOS FINAIS — TESTE RESERVADO")
print(f"MAE:  {mae_teste:.2f}")
print(f"RMSE: {rmse_teste:.2f}")
print(f"R²:   {r2_teste:.4f}")
print(f"Tempo de ajuste em desenvolvimento: {tempo_final / 60:.2f} minutos")

# --- GRÁFICO ---
rng = np.random.default_rng(42)
indices = rng.choice(len(y_teste), size=min(10000, len(y_teste)), replace=False)

plt.figure(figsize=(7, 5))
plt.scatter(np.asarray(y_teste)[indices], previsoes_teste[indices], alpha=0.30, s=14, color="#f28e2b", label="SVM")
plt.plot([y_teste.min(), y_teste.max()], [y_teste.min(), y_teste.max()], "--", color="#d1495b", label="Previsão perfeita")
plt.xlabel("Nota real de Linguagens")
plt.ylabel("Nota prevista")
plt.title("ENEM 2023 — Real vs Previsto (SVM)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()